In [0]:
source_dir="/Volumes/eccom_catalog/default/ecomm_data/products_data/Source/"
archive_dir="/Volumes/eccom_catalog/default/ecomm_data/products_data/Archive/"
product_stage_table="eccom_catalog.default.product_stage"
error_table="eccom_catalog.default.error_table"

In [0]:
#Read products csv file
from pyspark.sql.functions import col,current_timestamp,lit,current_date,datediff,when,split
from datetime import datetime
try:
    df_product=spark.read.csv(source_dir, header=True, inferSchema=True, dateFormat="yyyy-MM-dd", timestampFormat="yyyy-MM-dd HH:mm:ss")
    df_product=df_product.withColumn("processed_timestamp", current_timestamp())\
        .withColumn("batch_id", lit(datetime.now().strftime("%Y%m%d_%H%M%S")))\
        .withColumn("source_system", lit("ecommerce_products"))

    print("Successfully read products file")

except Exception as e:
    print(f"Failed to read products file: {str(e)}")
    raise

In [0]:
#Data Quality checks
try:
    total_records=df_product.count()
    total_null_product_ids=df_product.filter(col("product_id").isNull()).count()
    total_invalid_prices=df_product.filter(col("price")<=0).count()
    total_negative_stock=df_product.filter(col("stock_quantity")<0).count()
    total_future_dates=df_product.filter(col("launch_date")>current_date()).count()

    print(f"Total records: {total_records}")
    print(f"Total null product ids: {total_null_product_ids}")
    print(f"Total invalid prices: {total_invalid_prices}")
    print(f"Total negative stocks: {total_negative_stock}")
    print(f"Total future launch date products: {total_future_dates}")

    #Filter out valid records
    valid_records=df_product.filter((col("product_id").isNotNull()) & (col("price")>0) & (col("stock_quantity")>=0) 
                                    & (col("launch_date")<=current_date()))	
                                    
    invalid_records=df_product.filter((col("product_id").isNull()) | (col("price")<=0) | (col("stock_quantity")<0) | (col("launch_date")   >current_date()))
    total_valid_records=valid_records.count()
    total_invalid_records=invalid_records.count()

    print(f"Total valid records: {total_invalid_records}")
    print(f"Total invalid records: {total_invalid_records}")

except Exception as e:
       print(f"Failed to apply Data Quality checks: {str(e)}")		
       raise

In [0]:
#Data enrichment - Product Categorization and pricing analysis
from pyspark.sql.functions import size
try:
	product_segment=valid_records.withColumn("price_segment", when(col("price")<50, "Budget")
															  .when(col("price")<150, "Mid-range")
															  .when(col("price")<300, "Premium")
															  .otherwise("Luxury"))
													
	#Create stock status
	product_segment_status=product_segment.withColumn("stock_status", when(col("stock_quantity")==0, "Out of Stock")
																	   .when(col("stock_quantity")<10, "Low Stock")
																	   .when(col("stock_quantity")<50, "Medium Stock")
																	   .otherwise("High Stock"))
	#Calculate days since launch
	product_segment_status=product_segment_status.withColumn("days_since_launch", datediff(current_date(),col("launch_date")))	
	#Create product lifecycle stage
	product_lifecycle =	product_segment_status.withColumn("lifecycle_stage", when(col("days_since_launch")<30, "New")
																			 .when(col("days_since_launch")<365, "Growth")
																			 .when(col("discontinued")==True, "Discontinued")
																			 .otherwise("Mature"))
	#Parse dimensions and calculate volume 
	product_dimensions=product_lifecycle.withColumn("dimensions_array", split(col("dimensions_cm"),"x"))	
	#Calculate volume (assuming dimensions are in format "LXWXH")
	product_volume=product_dimensions.withColumn("volume_cm3", when(size(col("dimensions_array"))==3, col("dimensions_array")[0].cast ("double")*col("dimensions_array")[1].cast("double")*col("dimensions_array")[2].cast("double")).otherwise(0))
	#Calculate density weight/volume
	product_density=product_volume.withColumn("density_kg_per_cm3", when(col("volume_cm3")>0, col("weight_kg")/col("volume_cm3"))
																	.otherwise(0))
	print("Data enrichment completed")

except Exception as e:
    print(f"Failed in data enrichment: {str(e)}")
    raise

In [0]:
#Write valid data into staging table
try:
	product_density.write.format("delta").mode("overwrite").saveAsTable(product_stage_table)
	print(f"Successfully loaded {total_valid_records} valid products to staging table")	
	#Write invalid data into error table
	if total_invalid_records>0:
	   invalid_records.withColumn("error_reason", lit("Data quality validation fail")).withColumn("error_timestamp", current_timestamp())\
	   .write.format("delta").mode("append").saveAsTable(error_table)
	   print(f"Logged {total_invalid_records} invalid records to error table")
	else:
	   print("No invalid records found")   

except Exception as e:
    print(f"Error writing to staging table: {str(e)}")
    raise

In [0]:
#Archive processed files
try:
	files=dbutils.fs.ls(source_dir)	
	archive_count=0
	for file in files:
		if file.name.endswith(".csv"):
			source=file.path
			archive=archive_dir+file.name
			dbutils.fs.mv(source,archive)
			archive_count+=1
			print(f"Archived: {file.name}")
	print(f"Successfully archived {archive_count} files")

except Exception as e:
    print(f"Error archiving files: {str(e)}")
    raise 

In [0]:
#Log processing summary	
import json
try:
	processing_summary={
	"task": "products_stage_load",
	"timestamp": datetime.now().isoformat(),
	"total_records": total_records,
	"valid_records": total_valid_records,
	"invalid_records": total_invalid_records,
	"archived_files": archive_count,
	"status": "SUCCESS" if total_invalid_records==0 else "SUCCESS_WITH_WARNINGS"
	}
    
	print(f"Processing Summary: {json.dumps(processing_summary)}")
	processing_log=spark.createDataFrame([processing_summary])
	processing_log.write.format("delta").mode("append").saveAsTable("eccom_catalog.default.processing_log")
	print("logged archived files")

except Exception as e:
	print(f"Failed to logged archived files")
	raise